In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os

# print(os.listdir("/content/drive/MyDrive"))

Mounted at /content/drive


In [3]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q sentencepiece protobuf
!pip install -q pillow matplotlib seaborn pandas numpy tqdm scikit-learn
!pip install -U transformers
!pip install -U datasets
!pip install -U peft
!pip install -U trl
!pip install -U accelerate
!pip install -U bitsandbytes
!pip install -U pillow
!pip install -U sentencepiece
!pip install -U qwen-vl-utils

   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.6 MB 5.4 MB/s eta 0:00:03
   ----- ---------------------------------- 1.6/11.6 MB 5.3 MB/s eta 0:00:02
   --------- ------------------------------ 2.9/11.6 MB 5.7 MB/s eta 0:00:02
   -------------- ------------------------- 4.2/11.6 MB 5.7 MB/s eta 0:00:02
   ---------------- ----------------------- 4.7/11.6 MB 5.6 MB/s eta 0:00:02
   ---------------- ----------------------- 4.7/11.6 MB 5.6 MB/s eta 0:00:02
   ------------------ --------------------- 5.5/11.6 MB 4.2 MB/s eta 0:00:02
   --------------------- ------------------ 6.3/11.6 MB 4.0 MB/s eta 0:00:02
   ----------------------- ---------------- 6.8/11.6 MB 3.8 MB/s eta 0:00:02
   --------------------------- ------------ 7.9/11.6 MB 3.9 MB/s eta 0:00:01
   --------------------------- ------------ 8.1/11.6 MB 3.7 MB/s eta 0:00:01
   ----------------------------- ---------- 8.7/11.6 MB 3.5 MB/s eta 0:00:01
   ---

In [5]:
import os
import torch
from PIL import Image
from datasets import Dataset, concatenate_datasets
from torchvision import transforms

# ============================================================
# Dataset Path
# ============================================================
DATASET_PATH = "/content/drive/MyDrive/MLforHealthcaremimic-cxr"

# Output Folder
OUTPUT_DIR =  "/content/drive/MyDrive/preprocessed_dataset"
# ============================================================
# Load Dataset
# ============================================================

train_part1 = Dataset.from_file(
    os.path.join(DATASET_PATH, "mimic-cxr-train-00000-of-00002.arrow")
)

train_part2 = Dataset.from_file(
    os.path.join(DATASET_PATH, "mimic-cxr-train-00001-of-00002.arrow")
)

train_data = concatenate_datasets([train_part1, train_part2])

validation_data = Dataset.from_file(
    os.path.join(DATASET_PATH, "mimic-cxr-validation.arrow")
)

test_data = Dataset.from_file(
    os.path.join(DATASET_PATH, "mimic-cxr-test.arrow")
)

print("Dataset Loaded Successfully!")


FileNotFoundError: [WinError 3] Failed to open local file '/content/drive/MyDrive/MLforHealthcaremimic-cxr/mimic-cxr-train-00000-of-00002.arrow'. Detail: [Windows error 3] The system cannot find the path specified.


In [ ]:
# ============================================================
# Image Preprocessing
# ============================================================

from PIL import Image
import os

def preprocess_and_save(dataset, split_name):

    image_dir = os.path.join(OUTPUT_DIR, split_name, "images")
    report_dir = os.path.join(OUTPUT_DIR, split_name, "reports")

    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(report_dir, exist_ok=True)

    print(f"\nProcessing {split_name}...")

    for idx, sample in enumerate(dataset):

        # ---------------- Image ----------------
        image = sample["image"]

        if not isinstance(image, Image.Image):
            image = Image.open(image)

        # Convert to RGB
        image = image.convert("RGB")

        # Resize (optional but fixed size)
        image = image.resize((224, 224))

        # Save image
        image.save(
            os.path.join(image_dir, f"{idx:06d}.png"),
            format="PNG"
        )

        # ---------------- Report ----------------
        report = sample.get("reports", "")

        with open(
            os.path.join(report_dir, f"{idx:06d}.txt"),
            "w",
            encoding="utf-8"
        ) as f:
            f.write(report)

    print(f"{split_name} completed.")


# ============================================================
# Process All Splits
# ============================================================

preprocess_and_save(train_data, "train")
preprocess_and_save(validation_data, "validation")
preprocess_and_save(test_data, "test")

print("\nPreprocessing Completed Successfully!")


Processing train...
train completed.

Processing validation...
validation completed.

Processing test...
